# Phi：同じ文で質問形式を比較＋GitHub自動保存

T4 GPUで「すべてのセルを実行」。前回の72文×2質問=144入力です。新しいホールドアウト検証ではなく、質問による差を調べる探索です。

最後のセルで `ryouy/GraduationThesis` のmainに結果だけを新規保存します。Colabのシークレット名は初期値 `GITHUB_TOKEN`（`GH_TOKEN` / `github_token` / `github_pat` も自動確認）。名前が違う場合は下の `SECRET_NAME` だけ変更してください。秘密値はコードへ貼りません。シークレットのノートブックアクセスを許可してください。

保存先：`1104_phi_prompt_diagnostic/results/<run_id>/`。保存セルだけ再実行可能。保存が失敗しても結果ZIPはダウンロードできます。


In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install transformers==4.57.6 accelerate scikit-learn scipy pandas numpy sentencepiece requests matplotlib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, json, hashlib, torch
assert torch.cuda.is_available(), 'T4 GPUが利用できません。ランタイム設定を確認してください。'
print('GPU:',torch.cuda.get_device_name(0))
REPO=Path('/content/GraduationThesis-phi-prompt')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ryouy/GraduationThesis.git',str(REPO)],check=True)
else:
    subprocess.run(['git','pull','--ff-only'],cwd=REPO,check=True)
FOLDER=REPO/'1104_phi_prompt_diagnostic'
for name,digest in json.loads((FOLDER/'FROZEN_SHA256.json').read_text()).items():
    assert hashlib.sha256((FOLDER/name).read_bytes()).hexdigest()==digest,name
subprocess.run([sys.executable,str(FOLDER/'check.py')],cwd=REPO,check=True)
sys.path.insert(0,str(FOLDER))
from github_save import check_access, save_results, GitHubSaveError
SECRET_NAME='GITHUB_TOKEN' # キー名だけを指定。トークンの値は入力しません。
check_access(SECRET_NAME)
OUT=Path('/content')/('phi-prompt-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
print('Result directory:',OUT)


In [ ]:
subprocess.run([sys.executable,'-u',str(FOLDER/'run.py'),'--resolve','--out',str(OUT)],cwd=REPO,check=True)
subprocess.run([sys.executable,'-u',str(FOLDER/'run.py'),'--key','phi3_5mini','--out',str(OUT)],cwd=REPO,check=True)


In [ ]:
subprocess.run([sys.executable,str(FOLDER/'analyze.py'),str(OUT)],cwd=REPO,check=True)
from IPython.display import display, Image
display(Image(filename=str(OUT/'comparison.png')))
print('原回答の内容点検はまだ必要です。この比較だけでモデル採用は確定しません。')


In [ ]:
# 最終セル：検算済み結果をmainへ自動保存。その後ZIPもダウンロード。
# 保存に失敗した場合は、推論セルを再実行せず、このセルだけ再実行できます。
import shutil
from google.colab import files
try:
    receipt=save_results(OUT,SECRET_NAME)
    print('GitHub保存確認済み:',receipt['url'])
except GitHubSaveError as exc:
    print(str(exc))
    print('自動保存は完了していません。ZIPを保管し、設定を確認後にこのセルだけ再実行してください。')
finally:
    archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
    files.download(archive)
